In [0]:
%sql
-- Cross-system identity: CRM contacts.name (user_32digit hash) → mdl_users via ALL hash columns
-- USERNAME (86 matches), FIRSTNAME (2,809), LASTNAME (1,150) — UNION deduplicates overlaps
CREATE OR REPLACE TABLE crm_ingestion.gold.days_since_last_login AS (
WITH coaching_clients AS (
    SELECT id AS lead_id
    FROM crm_ingestion.silver.leads_processed
    WHERE status_label = 'Coaching Client'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC) = 1
),
crm_to_moodle AS (
    SELECT lead_id, LASTACCESS FROM (
        SELECT DISTINCT
            c.leads_processed_id AS lead_id,
            m.LASTACCESS
        FROM crm_ingestion.silver.leads_processed_contacts c
        JOIN crm_ingestion.silver.mdl_users_processed m ON c.name = m.USERNAME
        WHERE m.LASTACCESS IS NOT NULL AND m.LASTACCESS != ''
        UNION
        SELECT DISTINCT
            c.leads_processed_id AS lead_id,
            m.LASTACCESS
        FROM crm_ingestion.silver.leads_processed_contacts c
        JOIN crm_ingestion.silver.mdl_users_processed m ON c.name = m.FIRSTNAME
        WHERE m.LASTACCESS IS NOT NULL AND m.LASTACCESS != ''
        UNION
        SELECT DISTINCT
            c.leads_processed_id AS lead_id,
            m.LASTACCESS
        FROM crm_ingestion.silver.leads_processed_contacts c
        JOIN crm_ingestion.silver.mdl_users_processed m ON c.name = m.LASTNAME
        WHERE m.LASTACCESS IS NOT NULL AND m.LASTACCESS != ''
    )
    QUALIFY ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY LASTACCESS DESC) = 1
)
SELECT
    cc.lead_id,
    DATEDIFF(CURRENT_DATE, DATE(m.LASTACCESS)) AS days_since_last_login
FROM coaching_clients cc
JOIN crm_to_moodle m ON cc.lead_id = m.lead_id
)